# Fine-tune IndicF5 on your own voice — Kaggle T4

Reconstructed from a run that worked. Every cell below carries a fix for
something that blocked the first attempt; the comments say which.

**Before you start**

1. Phone-verify your Kaggle account (required for GPU).
2. Session options → **Internet ON _and_ GPU T4**.
   Toggling internet **resets the accelerator** — set internet first, then
   re-select the GPU and confirm both.
3. Accept the IndicF5 licence at huggingface.co/ai4bharat/IndicF5 and add
   your token as a Kaggle secret named `HF_TOKEN`.
4. Upload ~1–3 hours of clean single-speaker audio as a Kaggle Dataset.
   Zip it with Python's `zipfile`, **not** PowerShell `Compress-Archive` —
   that writes backslash entry names Kaggle rejects.

> **On how long to train:** see the note at the end. Training to convergence
> on ~1 hour of audio overwrites the base model's language knowledge. This
> is the single biggest trap in the whole process.


## 1 · Environment

`%pip`, not `!pip`. `!pip` installs into a different environment than the
kernel uses, and the imports then fail for no visible reason.


In [ ]:
%pip install -q f5-tts faster-whisper soundfile soxr

import torch, os, glob, shutil, gc
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
# if this says NO GPU, the internet toggle reset your accelerator - fix it now


## 2 · Find the dataset

Kaggle mounts datasets at a path that depends on your username and slug.
Glob for it instead of hard-coding.


In [ ]:
hits = glob.glob('/kaggle/input/**/wav_clean', recursive=True)
assert hits, 'wav_clean not found - check the dataset is attached'
WAV_DIR = hits[0]
print('audio :', WAV_DIR, '|', len(glob.glob(WAV_DIR + '/*.wav')), 'files')

WORK = '/kaggle/working'
DATASET_NAME = 'marathi_voice'


## 3 · Segment and transcribe

Whisper holds **3.8 GB of VRAM** and does not release it when the cell
ends. Training then OOMs for reasons that look unrelated. Free it
explicitly before moving on — this is fix #7 of 10.


In [ ]:
from faster_whisper import WhisperModel
import soundfile as sf

model = WhisperModel('large-v3', device='cuda', compute_type='float16')

OUT = f'{WORK}/clips'; os.makedirs(OUT, exist_ok=True)
rows = []
for wav in sorted(glob.glob(WAV_DIR + '/*.wav')):
    segs, _ = model.transcribe(wav, language='mr', vad_filter=True,
                               vad_parameters=dict(min_silence_duration_ms=400))
    audio, sr = sf.read(wav)
    for i, s in enumerate(segs):
        if not (1.0 < s.end - s.start < 20.0):
            continue                      # too short to learn from, too long to batch
        clip = audio[int(s.start * sr): int(s.end * sr)]
        name = f'{Path(wav).stem}_{i:04d}.wav'
        sf.write(f'{OUT}/{name}', clip, sr)
        rows.append((f'{OUT}/{name}', s.text.strip()))
print(len(rows), 'clips')

# fix #7: release Whisper's 3.8 GB or training will OOM
del model; gc.collect(); torch.cuda.empty_cache()
print('free VRAM:', torch.cuda.mem_get_info()[0] / 1e9, 'GB')


## 4 · metadata.csv

Two things `prepare_csv_wavs.py` is fussy about, both of which fail
unhelpfully:

- audio paths must be **absolute**
- you pass it the **path to metadata.csv**, not the directory it lives in

**Check the transcripts by hand if you can.** Whisper misspells Marathi
names, and whatever it writes is what the model learns to say.


In [ ]:
import csv
META = f'{WORK}/metadata.csv'
with open(META, 'w', encoding='utf-8', newline='') as f:
    w = csv.writer(f, delimiter='|')
    w.writerow(['audio_file', 'text'])
    for path, text in rows:
        w.writerow([os.path.abspath(path), text])   # absolute, not relative
print(open(META, encoding='utf-8').read()[:400])


In [ ]:
!python -m f5_tts.train.datasets.prepare_csv_wavs \
    {META} {WORK}/data/{DATASET_NAME}_char        # the FILE, not its folder


## 5 · Convert IndicF5 into an F5-TTS checkpoint

**The cell that took four attempts.** IndicF5 ships as safetensors with keys
prefixed `ema_model._orig_mod.` plus a bundled Vocos vocoder. The trainer
wants something quite specific:

| what | why |
|---|---|
| strip `_orig_mod.` | that is a `torch.compile` wrapper, not part of the model |
| drop `vocoder.*` | Vocos is loaded separately at inference |
| add `initted` / `step` | EMA bookkeeping the trainer reads on load |
| also save `model_state_dict` with **bare** keys | the non-EMA copy |
| **no top-level `step` / `update`** | those make the trainer take the *resume* branch and demand an optimizer state that does not exist |


In [ ]:
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from kaggle_secrets import UserSecretsClient

os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

src = hf_hub_download('ai4bharat/IndicF5', 'model.safetensors')
vocab_src = hf_hub_download('ai4bharat/IndicF5', 'checkpoints/vocab.txt')
raw = load_file(src)

ema, bare = {}, {}
for k, v in raw.items():
    if k.startswith('vocoder.'):
        continue                                   # Vocos, loaded separately
    clean = k.replace('_orig_mod.', '')            # drop the compile wrapper
    ema[clean] = v                                 # keeps the ema_model. prefix
    bare[clean.replace('ema_model.', '')] = v      # non-EMA copy, bare keys

ema['initted'] = torch.tensor(True)
ema['step'] = torch.tensor(0)

# NOTE: no top-level 'step' or 'update' key here, on purpose
ckpt = {'ema_model_state_dict': ema, 'model_state_dict': bare}
os.makedirs(f'{WORK}/base', exist_ok=True)
torch.save(ckpt, f'{WORK}/base/pretrained_indicf5.pt')
shutil.copy(vocab_src, f'{WORK}/base/vocab.txt')
print(len(ema), 'ema tensors,', len(bare), 'bare tensors')


## 6 · Stage the checkpoint

The finetune CLI will **not** overwrite an existing `pretrained_*.pt`, so a
stale copy from an earlier attempt silently trains the wrong weights. Clear
it. IndicF5's vocab is 2545 chars and must replace the dataset's own.


In [ ]:
CK = f'{WORK}/ckpts/{DATASET_NAME}'
shutil.rmtree(CK, ignore_errors=True)          # fix #8: never trust a stale copy
os.makedirs(CK, exist_ok=True)
shutil.copy(f'{WORK}/base/pretrained_indicf5.pt', f'{CK}/pretrained_indicf5.pt')
shutil.copy(f'{WORK}/base/vocab.txt', f'{WORK}/data/{DATASET_NAME}_char/vocab.txt')
print(os.listdir(CK))


## 7 · Train

**Disk.** Each checkpoint is ~5.4 GB — model + EMA + optimizer, not the
1.4 GB the model size suggests. Kaggle's working disk is ~20 GB and fills
around epoch 16 with default retention. The flags below keep one.

**VRAM.** `BATCH_FRAMES 2400` fits a T4 *after* Whisper is freed. 3200 OOMs.

**Epochs — read this.** 80 epochs over ~1 hour of audio learns the voice and
**destroys the base model's language knowledge**. See the closing note.


In [ ]:
!accelerate launch -m f5_tts.train.finetune_cli \
  --exp_name F5TTS_Base \
  --dataset_name {DATASET_NAME} \
  --finetune True \
  --pretrain {CK}/pretrained_indicf5.pt \
  --tokenizer char \
  --learning_rate 1e-5 \
  --batch_size_per_gpu 2400 \
  --batch_size_type frame \
  --max_samples 64 \
  --epochs 30 \
  --save_per_updates 8000 \
  --last_per_updates 1000 \
  --keep_last_n_checkpoints 1


## 8 · Pick a checkpoint and sanity-check it

Exclude `pretrained_*.pt` — it sorts after `model_*.pt` and you will pick it
by accident. Generate one line and listen before downloading 5 GB.

**If the audio is silent, do not assume the model failed.** In float16 this
model emits `NaN`, and libsndfile writes NaN as a constant −1.0 — a flat DC
line with the right duration and no sound. Force float32.


In [ ]:
import numpy as np
cands = [p for p in glob.glob(f'{CK}/model_*.pt') if 'pretrained' not in p]
best = sorted(cands)[-1]
print('using', best)

from f5_tts.api import F5TTS
tts = F5TTS(model='F5TTS_Base', ckpt_file=best,
            vocab_file=f'{WORK}/base/vocab.txt', device='cuda')
for m in ('ema_model', 'model', 'vocoder'):
    if hasattr(tts, m):
        getattr(tts, m).float()          # float16 -> NaN -> silent audio

wav, sr, _ = tts.infer(ref_file=rows[0][0], ref_text=rows[0][1],
                       gen_text='ही एक चाचणी आहे.', nfe_step=32)
wav = np.asarray(wav, dtype=np.float32)
print('peak %.2f dBFS | NaN=%s' % (20*np.log10(max(abs(wav).max(),1e-9)), np.isnan(wav).any()))
from IPython.display import Audio; Audio(wav, rate=sr)


## 9 · Slim and download

Dropping the optimizer state halves the file with no effect on inference.

> Your browser will save it as **`model_last.zip`**. That file already *is*
> the `.pt` — a PyTorch checkpoint is a zip container. **Rename it, do not
> extract it.** Extracting gives `data.pkl` and numbered fragments.


In [ ]:
slim = {k: v for k, v in torch.load(best, map_location='cpu').items()
        if k in ('ema_model_state_dict', 'model_state_dict')}
torch.save(slim, f'{WORK}/model_last_slim.pt')
print('%.2f GB' % (os.path.getsize(f'{WORK}/model_last_slim.pt') / 1e9))
# download model_last_slim.pt AND base/vocab.txt from the Output tab


---
## The thing that matters most

A fine-tune that trains to convergence on ~1 hour of one voice will come
back with **the right voice and broken language**. In this project it read
`थांबलं` as `थांबला`, put a `ळ` in `कोल्हापूर`, and dropped a syllable from a
name — while sounding convincingly like the speaker.

That is **catastrophic forgetting**. IndicF5 knew those words from hundreds
of hours; 80 epochs over 1.26 hours overwrote that knowledge.

Two ways to deal with it:

**Prevent it** — train less. Far fewer epochs, a lower learning rate, or
LoRA so the base weights barely move. The `--epochs 30` above is already
reduced from the 80 that caused the damage; stop earlier still if the voice
is recognisable.

**Repair it** — blend the fine-tune back toward the base afterwards:

```bash
python tools/merge_ckpt.py --base indicf5_base.pt \
                           --tuned model_last_slim.pt --sweep
```

`merged = α · finetuned + (1 − α) · base`. Speaker identity lives in a small
part of the weights while pronunciation is spread across all of them, so a
partial blend keeps the voice and recovers the language. **α = 0.5** was the
pick here: pronunciation correct, voice ~90% intact.

Full reasoning: [WIKI.md §9](../WIKI.md#9-catastrophic-forgetting--the-big-one).

---

*Reconstructed from a working run's debug log rather than exported from
Kaggle, so treat cell outputs as illustrative. The fixes are the parts that
were verified.*
